# Simple OpenAI chatbot (LangChain)

Uses **LangChain** (`ChatOpenAI`), a **dropdown** to pick the model, and **text widgets** for your messages (Enter or **Send**). Run all cells once, then use the UI below.

The notebook loads **`.env`** and **`OPENAI_API_KEY`** (or use a shell-exported key).

In [1]:
import os
import subprocess
import sys

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    # %pip targets THIS notebook kernel (fixes wrong global Python in VS Code / Jupyter).
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None:
            ip.run_line_magic("pip", "install -q python-dotenv")
        else:
            raise RuntimeError("no ipython")
    except Exception:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "python-dotenv"]
        )
    from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Uncomment it in `.env` or export it before starting Jupyter."
    )

print("API key loaded.")

API key loaded.


In [2]:
from typing import Any

import ipywidgets as widgets
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# ---------------------------------------------------------------------------
# Per-model settings (used when you pick a model in the dropdown)
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = "You are a helpful assistant."

MODEL_CONFIGS: dict[str, dict[str, Any]] = {
    "gpt-4o-mini": {"temperature": 0.7, "max_tokens": 1024},
    "gpt-4o": {"temperature": 0.7, "max_tokens": 2048},
    "gpt-3.5-turbo": {"temperature": 0.8, "max_tokens": 1024},
    "o3-mini": {
        "model_kwargs": {"reasoning_effort": "medium"},
        "max_completion_tokens": 4096,
    },
    "o1": {
        "model_kwargs": {"reasoning_effort": "high"},
        "max_completion_tokens": 8192,
    },
}

MODEL_CHOICES: list[tuple[str, str]] = [
    ("GPT-4o mini (fast)", "gpt-4o-mini"),
    ("GPT-4o", "gpt-4o"),
    ("GPT-3.5 Turbo (legacy)", "gpt-3.5-turbo"),
    ("o3-mini (reasoning)", "o3-mini"),
    ("o1 (reasoning)", "o1"),
]


def _is_reasoning_model(model: str) -> bool:
    return model.lower().startswith(("o1", "o3", "o4"))


def build_chat_model(model_name: str) -> ChatOpenAI:
    cfg = MODEL_CONFIGS.get(model_name, {})
    params: dict[str, Any] = {"model": model_name}
    if _is_reasoning_model(model_name):
        params["temperature"] = None
        mk = dict(cfg.get("model_kwargs") or {})
        if mk:
            params["model_kwargs"] = mk
        mct = cfg.get("max_completion_tokens")
        if mct is not None:
            params["max_tokens"] = mct
    else:
        if cfg.get("temperature") is not None:
            params["temperature"] = cfg["temperature"]
        if cfg.get("max_tokens") is not None:
            params["max_tokens"] = cfg["max_tokens"]
    return ChatOpenAI(**params)


def initial_messages(model_name: str) -> tuple[list[BaseMessage], bool]:
    """Returns (history, reasoning_needs_prefix_on_next_user_turn)."""
    if _is_reasoning_model(model_name):
        return [], True
    return [SystemMessage(content=SYSTEM_PROMPT)], False

In [3]:
model_dd = widgets.Dropdown(
    options=MODEL_CHOICES,
    value="gpt-4o-mini",
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(min_width="320px"),
)

line_input = widgets.Text(
    value="",
    placeholder="One-line message — press Enter to send",
    description="Quick:",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "initial"},
)

user_text = widgets.Textarea(
    value="",
    placeholder="Or use this box for longer / multi-line messages, then click Send.",
    description="You:",
    layout=widgets.Layout(width="100%", height="90px"),
    style={"description_width": "initial"},
)

send_btn = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear chat", button_style="warning")

status = widgets.HTML("")

transcript = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ccc",
        min_height="260px",
        max_height="480px",
        overflow_y="auto",
        width="100%",
        padding="8px",
    )
)

history: list[BaseMessage] = []
reasoning_prefix_pending = False


def _reset_for_model(_=None):
    global history, reasoning_prefix_pending
    history, reasoning_prefix_pending = initial_messages(model_dd.value)
    transcript.clear_output(wait=True)
    status.value = f"<i>New chat — model: <b>{model_dd.value}</b></i>"


def _append_turn(user_msg: str, assistant_msg: str) -> None:
    with transcript:
        display(Markdown(f"**You:** {user_msg}"))
        display(Markdown(f"**Assistant:** {assistant_msg}"))
        display(Markdown("---"))


def _on_send(_=None):
    global history, reasoning_prefix_pending
    raw = (user_text.value or "").strip()
    if raw:
        user_text.value = ""
    else:
        raw = (line_input.value or "").strip()
        line_input.value = ""
    if not raw:
        status.value = "<i>Enter a message first.</i>"
        return

    model_id = model_dd.value

    display_user = raw
    send_text = raw
    if reasoning_prefix_pending:
        send_text = f"{SYSTEM_PROMPT}\n\n{raw}"
        reasoning_prefix_pending = False

    llm = build_chat_model(model_id)
    turn = [*history, HumanMessage(content=send_text)]

    status.value = "<i>Thinking…</i>"
    try:
        out = llm.invoke(turn)
    except Exception as e:
        status.value = f"<span style='color:red'><b>Error:</b> {e}</span>"
        return

    reply = (out.content or "").strip() if isinstance(out.content, str) else str(out.content)
    history = [*turn, AIMessage(content=reply)]
    _append_turn(display_user, reply)
    status.value = f"<i>Model: <b>{model_id}</b></i>"


def _on_clear(_=None):
    _reset_for_model()


def _on_model_change(change):
    if change["name"] != "value":
        return
    if change.get("old") == change.get("new"):
        return
    _reset_for_model()


model_dd.observe(_on_model_change, names="value")
send_btn.on_click(_on_send)
line_input.on_submit(_on_send)
clear_btn.on_click(_on_clear)

_reset_for_model()

ui = widgets.VBox(
    [
        widgets.HTML("<h3>Chat</h3>"),
        model_dd,
        status,
        transcript,
        line_input,
        user_text,
        widgets.HBox([send_btn, clear_btn]),
        widgets.HTML(
            "<small>Tip: changing the model clears the transcript and starts a new conversation.</small>"
        ),
    ],
    layout=widgets.Layout(width="100%", max_width="720px"),
)

display(ui)

/var/folders/fg/s0jy31yn27j1x67qv69f6p_m0000gp/T/ipykernel_24748/3213510240.py:109: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  line_input.on_submit(_on_send)
